# Gas Prices — Data Preparation
Converts yearly xlsx/xls files into a single cleaned CSV with dates as index and cities as columns.

**Folder structure expected:**
```
data/
  raw/        ← place all xlsx/xls source files here
  processed/  ← cleaned CSV will be saved here (created automatically)
```

In [7]:
import pandas as pd
import re
from pathlib import Path

BASE_DIR       = Path().resolve()
raw_folder     = BASE_DIR / 'data' / 'raw'
processed_folder = BASE_DIR / 'data' / 'processed'

# Create processed folder if it doesn't exist
processed_folder.mkdir(parents=True, exist_ok=True)

print(f'Raw data     : {raw_folder}  (exists: {raw_folder.exists()})')
print(f'Processed    : {processed_folder}  (exists: {processed_folder.exists()})')

Raw data     : /Users/btosi/projects/gas_price/data/raw  (exists: True)
Processed    : /Users/btosi/projects/gas_price/data/processed  (exists: True)


## Step 1 — Helper functions

In [8]:
def is_date(val):
    """Match M/D format (e.g. 1/7) or full datetime strings (e.g. 2026-01-07 00:00:00)"""
    val = str(val).strip()
    return bool(re.match(r'^\d{1,2}/\d{1,2}$', val)) or bool(re.match(r'^\d{4}-\d{2}-\d{2}', val))

def parse_index(index, year):
    """Handle both M/D and full datetime formats, forcing the correct year."""
    parsed = []
    for val in index:
        val = str(val).strip()
        if re.match(r'^\d{1,2}/\d{1,2}$', val):
            parsed.append(pd.to_datetime(f'{val}/{year}', format='%m/%d/%Y'))
        elif re.match(r'^\d{4}-\d{2}-\d{2}', val):
            parsed.append(pd.to_datetime(val).replace(year=int(year)))
        else:
            parsed.append(pd.NaT)
    return pd.DatetimeIndex(parsed)

def check_columns(transposed, reference_year='2023'):
    """Compare columns across all years against a reference year."""
    all_columns = {year: set(df_t.columns) for year, df_t in transposed.items()}
    reference_cols = all_columns[reference_year]
    for year, cols in sorted(all_columns.items()):
        missing = reference_cols - cols
        extra   = cols - reference_cols
        if missing or extra:
            print(f'⚠️  {year} — missing: {missing} | extra: {extra}')
        else:
            print(f'✅  {year} — columns match')

## Step 2 — Load, transpose, clean and parse dates

In [9]:
GTA_NEIGHBORHOODS = ['NORTH YORK', 'SCARBOROUGH', 'ETOBICOKE', 'BRAMPTON', 'MISSISSAUGA', 'SARNIA', 'VAUGHAN/MARKHAM']
PEEL_COLS         = ['BRAMPTON', 'MISSISSAUGA']
JUNK_COLS         = ['nan', 'S-Simple V-Volume Weighted P-Population Weighted', 'Average']

RENAME_2014 = {
    'WINNIPEG\xa0\xa0'         : 'WINNIPEG',
    'SAINTJOHN'                 : 'SAINT JOHN',
    'STJOHNS'                   : 'ST JOHNS',
    'Large Markets Average (P)' : 'Large Markets Ave(P)',
    'Large Markets Average (S)' : 'Large Markets Ave(S)',
    'Small Markets Average (P)' : 'Small Markets Ave(P)',
    'Small Markets Average (S)' : 'Small Markets Ave(S)',
    'Atlantic Average (P)'      : 'Atlantic Ave(P)',
    'Québec Average (P)'        : 'Quebec Ave(P)',
    'Ontario Average (P)'       : 'Ontario Ave(P)',
    'Western Average (P)'       : 'Western Ave(P)',
    'Canada Ave (V)'            : 'Canada Ave(V)',
    'CANADA (P)'                : 'Canada(P)',
    'CANADA (S)'                : 'Canada(S)',
}

transposed = {}

for filepath in sorted(raw_folder.glob('*.xls*')):
    year = filepath.stem.split('_')[-1]

    # Convert xlsx/xls to CSV in raw folder
    df_raw = pd.read_excel(filepath)
    csv_path = raw_folder / filepath.with_suffix('.csv').name
    df_raw.to_csv(csv_path, index=False)

    # Load with correct header based on format
    if year in ['2015', '2016']:
        df = pd.read_csv(csv_path, index_col=0)           # Format A: dates already in header
    else:
        df = pd.read_csv(csv_path, header=2, index_col=0) # Format B: real header is at row 2

    # Transpose: cities become columns, dates become index
    df_t = df.T

    # Clean column names
    df_t.columns = df_t.columns.str.replace('*', '', regex=False).str.strip()

    # Drop date-named columns (artefact from multi-table layouts e.g. 2014)
    df_t = df_t[[c for c in df_t.columns if not is_date(str(c))]]

    # Drop junk columns
    df_t = df_t.drop(columns=[c for c in df_t.columns if str(c) in JUNK_COLS], errors='ignore')

    # Standardize city names
    if year == '2014':
        df_t = df_t.rename(columns=RENAME_2014)
    if 'TORONTO' in df_t.columns:
        df_t = df_t.rename(columns={'TORONTO': 'CITY OF TORONTO'})

    # Compute PEEL REGION before dropping neighborhoods
    existing_peel = [c for c in PEEL_COLS if c in df_t.columns]
    if existing_peel:
        df_t['PEEL REGION'] = df_t[existing_peel].astype(float).mean(axis=1)

    # Drop GTA neighborhoods
    df_t = df_t.drop(columns=[c for c in GTA_NEIGHBORHOODS if c in df_t.columns], errors='ignore')

    # Keep only date-like rows, parse, drop any NaT
    df_t = df_t[df_t.index.map(is_date)]
    df_t.index = parse_index(df_t.index, year)
    df_t = df_t[df_t.index.notna()]
    df_t.index.name = 'date'

    # Coerce all price values to numeric
    df_t = df_t.apply(pd.to_numeric, errors='coerce')

    transposed[year] = df_t
    print(f'✅ {year} — rows: {len(df_t)}, cols: {len(df_t.columns)}, sample: {df_t.index[:2].tolist()}')

✅ 2014 — rows: 52, cols: 71, sample: [Timestamp('2014-01-07 00:00:00'), Timestamp('2014-01-14 00:00:00')]
✅ 2015 — rows: 52, cols: 77, sample: [Timestamp('2015-01-06 00:00:00'), Timestamp('2015-01-13 00:00:00')]
✅ 2016 — rows: 52, cols: 81, sample: [Timestamp('2016-01-05 00:00:00'), Timestamp('2016-01-12 00:00:00')]
✅ 2017 — rows: 52, cols: 82, sample: [Timestamp('2017-01-03 00:00:00'), Timestamp('2017-01-10 00:00:00')]
✅ 2018 — rows: 52, cols: 82, sample: [Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-09 00:00:00')]


/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's defaul

✅ 2019 — rows: 53, cols: 82, sample: [Timestamp('2019-01-01 00:00:00'), Timestamp('2019-01-08 00:00:00')]
✅ 2020 — rows: 52, cols: 82, sample: [Timestamp('2020-01-07 00:00:00'), Timestamp('2020-01-14 00:00:00')]
✅ 2021 — rows: 52, cols: 82, sample: [Timestamp('2021-01-05 00:00:00'), Timestamp('2021-01-12 00:00:00')]
✅ 2022 — rows: 52, cols: 82, sample: [Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-11 00:00:00')]
✅ 2023 — rows: 52, cols: 82, sample: [Timestamp('2023-01-03 00:00:00'), Timestamp('2023-01-10 00:00:00')]


/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/btosi/projects/.conda-envs/ds-main/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ 2024 — rows: 53, cols: 82, sample: [Timestamp('2024-01-02 00:00:00'), Timestamp('2024-01-09 00:00:00')]
✅ 2025 — rows: 52, cols: 82, sample: [Timestamp('2025-01-07 00:00:00'), Timestamp('2025-01-14 00:00:00')]
✅ 2026 — rows: 52, cols: 82, sample: [Timestamp('2026-01-06 00:00:00'), Timestamp('2026-01-13 00:00:00')]


## Step 3 — Column check
Mismatches are cities that didn't exist in older datasets — expected, handled by `pd.concat` with NaN fill.

In [10]:
if not transposed:
	print("⚠️ transposed is empty. Run the data loading cell first.")
else:
	ref_year = '2023' if '2023' in transposed else sorted(transposed.keys())[-1]
	print(f"Using reference year: {ref_year}")
	check_columns(transposed, reference_year=ref_year)

Using reference year: 2023
⚠️  2014 — missing: {'BARRIE', 'GRANDE PRAIRIE', 'KITCHENER', 'ABBOTSFORD', 'GRAND FALLS', 'MOOSE JAW', 'OSHAWA', 'GATINEAU', 'BRANTFORD', 'GUELPH', 'PEEL REGION'} | extra: set()
⚠️  2015 — missing: {'GRANDE PRAIRIE', 'GRAND FALLS', 'MOOSE JAW', 'GATINEAU', 'PEEL REGION'} | extra: set()
⚠️  2016 — missing: {'PEEL REGION'} | extra: set()
✅  2017 — columns match
✅  2018 — columns match
✅  2019 — columns match
✅  2020 — columns match
✅  2021 — columns match
✅  2022 — columns match
✅  2023 — columns match
✅  2024 — columns match
✅  2025 — columns match
✅  2026 — columns match


## Step 4 — Combine and save

In [11]:
df_combined = pd.concat(transposed.values()).sort_index()

# Final safety pass — drop any column that still looks like a date
date_cols = [c for c in df_combined.columns if is_date(str(c))]
if date_cols:
    print(f'⚠️  Dropping {len(date_cols)} date-named columns: {date_cols}')
    df_combined = df_combined.drop(columns=date_cols)
else:
    print('✅ No date-named columns found.')

# Report columns with >50% NaN
nan_report = df_combined.isna().mean().sort_values(ascending=False)
high_nan   = nan_report[nan_report > 0.5]
if not high_nan.empty:
    print('\nColumns with >50% NaN (likely missing in older years — expected):')
    for col, pct in high_nan.items():
        print(f'   {col}: {pct:.0%} NaN')

print(f'\nShape     : {df_combined.shape}')
print(f'Date range: {df_combined.index.min()} → {df_combined.index.max()}')
print(f'Columns   : {len(df_combined.columns)}')
df_combined.head(3)

✅ No date-named columns found.

Shape     : (678, 82)
Date range: 2014-01-07 00:00:00 → 2026-12-29 00:00:00
Columns   : 82


,WHITEHORSE,VANCOUVER,VICTORIA,PRINCE GEORGE,KAMLOOPS,KELOWNA,FORT ST. JOHN,YELLOWKNIFE,CALGARY,RED DEER,...,BARRIE,BRANTFORD,GUELPH,KITCHENER,OSHAWA,GRANDE PRAIRIE,MOOSE JAW,GATINEAU,GRAND FALLS,PEEL REGION
date,,,,,,,,,,,,,,,,,,,,,
2014-01-07,127.9,129.3,119.4,112.9,115.7,123.9,132.9,138.9,109.2,103.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-01-14,127.9,129.4,111.4,114.9,113.0,119.9,129.9,138.9,107.8,103.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-01-21,127.9,129.3,120.9,120.7,115.9,125.9,132.4,138.9,106.7,103.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
output_path = processed_folder / 'gas_prices_all_years.csv'
df_combined.to_csv(output_path)
print(f'✅ Saved to {output_path}')

✅ Saved to /Users/btosi/projects/gas_price/data/processed/gas_prices_all_years.csv


In [13]:
# check all years  inside  the column date and do a count of how many dates are in each year, to see if any years are missing a lot of data
df_combined['year'] = df_combined.index.year
year_counts = df_combined['year'].value_counts().sort_index()
print('\n📅 Date counts by year:')
print(year_counts)


📅 Date counts by year:
year
2014    52
2015    52
2016    52
2017    52
2018    52
2019    53
2020    52
2021    52
2022    52
2023    52
2024    53
2025    52
2026    52
Name: count, dtype: int64


In [ ]:
#